In [1]:
from transformers import AutoTokenizer, AutoModel
from scipy.spatial.distance import cosine

/home/jagan/Documents/projects/LLM_ENGINEERING/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "bert-base-uncased"

model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [3]:
def predict(text):
    encoded_input = tokenizer(text, return_tensors='pt') #pr stance for PyTorch and TensorFlow use 'tf' and np for NumPy
    output = model(**encoded_input)
    return output.last_hidden_state

In [4]:
sentace_1 = "i eat apple"
sentacne_2 = "i baught new apple iphone"
token_1 = tokenizer.tokenize(sentace_1)
print("token 1:", token_1)
token_2 = tokenizer.tokenize(sentacne_2)
print("token 2:", token_2)


token 1: ['i', 'eat', 'apple']
token 2: ['i', 'ba', '##ught', 'new', 'apple', 'iphone']


In [5]:
out1 = predict(sentace_1)
out2 = predict(sentacne_2)
print("output 1 shape:", out1.shape)
print("output 2 shape:", out2.shape)
print("output 1:", out1)
print("output 2:", out2)    

output 1 shape: torch.Size([1, 5, 768])
output 2 shape: torch.Size([1, 8, 768])
output 1: tensor([[[-0.0041,  0.3931,  0.1125,  ...,  0.0533,  0.2567,  0.2992],
         [ 0.0553,  0.5783,  0.1838,  ..., -0.4128,  0.6033, -0.3740],
         [ 0.2442,  0.6832,  0.2792,  ...,  0.1786,  0.2229, -0.5779],
         [-0.4030, -0.0195, -0.2932,  ...,  0.7506,  0.3043, -0.4378],
         [ 0.8079,  0.4284,  0.0708,  ...,  0.0949, -0.3760, -0.4165]]],
       grad_fn=<NativeLayerNormBackward0>)
output 2: tensor([[[-0.3887,  0.2547,  0.2855,  ..., -0.2322,  0.1565,  0.1126],
         [-0.4536,  0.1929,  0.2084,  ..., -0.6935,  0.7364, -0.2046],
         [ 0.1019,  0.0649,  0.5578,  ..., -0.7750,  0.0118,  1.5456],
         ...,
         [ 0.3274,  1.0349,  0.5477,  ..., -0.4442, -0.0838, -1.5012],
         [-0.0689,  0.5958, -0.0129,  ...,  0.0741, -0.0549, -1.2561],
         [ 0.4275,  0.2748,  0.0106,  ...,  0.2540, -0.3523, -0.2548]]],
       grad_fn=<NativeLayerNormBackward0>)


In [6]:
embedding_1 = out1[0:, token_1.index('apple'), :].detach()
embedding_2 = out2[0:, token_2.index('apple'), :].detach()
cosine_distance = cosine(embedding_1.numpy().flatten(), embedding_2.numpy().flatten())
print("Cosine distance between embeddings:", cosine_distance)

Cosine distance between embeddings: 0.59797287


In [7]:
# Apply mean pooling to reduce embeddings to 1-D
embedding_1_pooled = embedding_1.mean(dim=0)
embedding_2_pooled = embedding_2.mean(dim=0)

# Calculate cosine distance
cosine_distance = cosine(embedding_1_pooled.numpy(), embedding_2_pooled.numpy())
print("Cosine distance between embeddings:", cosine_distance)

# Calculate cosine similarity
cosine_similarity = 1 - cosine_distance
print("Cosine similarity between embeddings:", cosine_similarity)

Cosine distance between embeddings: 0.59797287
Cosine similarity between embeddings: 0.40202713


In [9]:
# Case 1: Identical Context
sent1_identical = "I eat apple"
sent2_identical = "I love eating apple"

out1_identical = predict(sent1_identical)
out2_identical = predict(sent2_identical)

embedding_1_identical = out1_identical[0:, tokenizer.tokenize(sent1_identical).index('apple'), :].detach()
embedding_2_identical = out2_identical[0:, tokenizer.tokenize(sent2_identical).index('apple'), :].detach()

cosine_distance_identical = cosine(embedding_1_identical.numpy().flatten(), embedding_2_identical.numpy().flatten())
cosine_similarity_identical = 1 - cosine_distance_identical

print("Case 1 - Identical Context:")
print("Cosine Distance:", cosine_distance_identical)
print("Cosine Similarity:", cosine_similarity_identical)

# Case 2: Non-Identical Context
sent1_non_identical = "I eat apple"
sent2_non_identical = "I bought a new apple iPhone"

out1_non_identical = predict(sent1_non_identical)
out2_non_identical = predict(sent2_non_identical)

embedding_1_non_identical = out1_non_identical[0:, tokenizer.tokenize(sent1_non_identical).index('apple'), :].detach()
embedding_2_non_identical = out2_non_identical[0:, tokenizer.tokenize(sent2_non_identical).index('apple'), :].detach()

cosine_distance_non_identical = cosine(embedding_1_non_identical.numpy().flatten(), embedding_2_non_identical.numpy().flatten())
cosine_similarity_non_identical = 1 - cosine_distance_non_identical

print("\nCase 2 - Non-Identical Context:")
print("Cosine Distance:", cosine_distance_non_identical)
print("Cosine Similarity:", cosine_similarity_non_identical)

Case 1 - Identical Context:
Cosine Distance: 0.32067287
Cosine Similarity: 0.67932713

Case 2 - Non-Identical Context:
Cosine Distance: 0.6726324
Cosine Similarity: 0.3273676


In [10]:
# Implement Masked Language Modeling (MLM)
from torch.nn.functional import softmax
import torch
 
# Define a sentence with a masked token
masked_sentence = "I love [MASK] apple. It is delicious."
 
# Tokenize the sentence with the mask token
encoded_input = tokenizer(masked_sentence, return_tensors='pt')
mask_token_index = torch.where(encoded_input['input_ids'] == tokenizer.mask_token_id)[1]
 
# Get the model output
output = model(**encoded_input)
logits = output.logits
 
# Extract logits for the masked token
mask_token_logits = logits[0, mask_token_index, :]
 
# Apply softmax to get probabilities
mask_token_probs = softmax(mask_token_logits, dim=-1)
 
# Get the predicted token
predicted_token_id = torch.argmax(mask_token_probs, dim=-1)
predicted_token = tokenizer.decode(predicted_token_id)
 
# Print the logits, probabilities, and predicted token
print("Logits for masked token:", mask_token_logits)
print("Probabilities for masked token:", mask_token_probs)
print("Predicted token:", predicted_token)

AttributeError: 'BaseModelOutputWithPoolingAndCrossAttentions' object has no attribute 'logits'

In [12]:
# Update to use AutoModelForMaskedLM for MLM tasks
from transformers import AutoModelForMaskedLM
 
# Load the model for Masked Language Modeling
model_name = "bert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
 
# Define a sentence with a masked token
masked_sentence = "I love [MASK] apple. It is delicious."
 
# Tokenize the sentence with the mask token
encoded_input = tokenizer(masked_sentence, return_tensors='pt')
mask_token_index = torch.where(encoded_input['input_ids'] == tokenizer.mask_token_id)[1]
 
# Get the model output
output = model(**encoded_input)
logits = output.logits
print("Logits shape:", logits.shape,logits)
 
# Extract logits for the masked token
mask_token_logits = logits[0, mask_token_index, :]
 
# Apply softmax to get probabilities
mask_token_probs = softmax(mask_token_logits, dim=-1)
 
# Get the predicted token
predicted_token_id = torch.argmax(mask_token_probs, dim=-1)
predicted_token = tokenizer.decode(predicted_token_id)
 
# Print the logits, probabilities, and predicted token
print("Logits for masked token:", mask_token_logits)
print("Probabilities for masked token:", mask_token_probs)
print("Predicted token:", predicted_token)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Logits shape: torch.Size([1, 11, 30522]) tensor([[[ -6.7390,  -6.6858,  -6.7035,  ...,  -6.0940,  -5.8939,  -4.0284],
         [-10.0607,  -9.8693,  -9.9823,  ..., -10.1048,  -9.0438,  -6.2466],
         [-10.8222, -10.9683, -11.1529,  ..., -11.5972, -10.2376,  -9.5010],
         ...,
         [ -3.4192,  -3.3048,  -3.6517,  ...,  -3.3489,  -2.9290,  -2.0261],
         [-13.2872, -13.0725, -13.3696,  ..., -11.7374, -11.3738,  -7.1374],
         [-12.5476, -12.4854, -12.4648,  ..., -12.4908, -11.8247,  -7.2815]]],
       grad_fn=<ViewBackward0>)
Logits for masked token: tensor([[-9.2731, -8.9453, -9.2232,  ..., -9.0469, -7.1632, -8.8412]],
       grad_fn=<IndexBackward0>)
Probabilities for masked token: tensor([[1.2637e-09, 1.7539e-09, 1.3284e-09,  ..., 1.5845e-09, 1.0422e-08,
         1.9464e-09]], grad_fn=<SoftmaxBackward0>)
Predicted token: the


In [13]:
# Get the top 10 predicted tokens with probabilities
top_10_probs, top_10_token_ids = torch.topk(mask_token_probs, 10, dim=-1)
top_10_tokens = [tokenizer.decode([token_id]) for token_id in top_10_token_ids[0]]
 
# Print the top 10 tokens with their probabilities
print("Top 10 predicted tokens with probabilities:")
for token, prob in zip(top_10_tokens, top_10_probs[0]):
    print(f"{token}: {prob.item():.4f}")

Top 10 predicted tokens with probabilities:
the: 0.4067
this: 0.3109
that: 0.1284
my: 0.0375
your: 0.0308
an: 0.0163
his: 0.0078
her: 0.0059
red: 0.0050
green: 0.0037


In [14]:
# Add the top 10 predicted words to the sentence and print each variation
print("\nTop 10 sentence variations:")
for token, prob in zip(top_10_tokens, top_10_probs[0]):
    modified_sentence = masked_sentence.replace("[MASK]", token)
    print(f"{modified_sentence} (Probability: {prob.item():.4f})")


Top 10 sentence variations:
I love the apple. It is delicious. (Probability: 0.4067)
I love this apple. It is delicious. (Probability: 0.3109)
I love that apple. It is delicious. (Probability: 0.1284)
I love my apple. It is delicious. (Probability: 0.0375)
I love your apple. It is delicious. (Probability: 0.0308)
I love an apple. It is delicious. (Probability: 0.0163)
I love his apple. It is delicious. (Probability: 0.0078)
I love her apple. It is delicious. (Probability: 0.0059)
I love red apple. It is delicious. (Probability: 0.0050)
I love green apple. It is delicious. (Probability: 0.0037)
